# Case 8 — RAG Pipeline

Case 7, kural motorunun kararlarını yapılandırılmış (rule_id, severity, koşul, şablon mesaj)
biçimde açıkladı. Case 8, bunu **bilgi destekli, doğal dilde** açıklanabilirliğe taşıyor: örnek
fraud-policy dökümanlarından bir knowledge base kuruyoruz, bir soru/işlem için ilgili policy
metnini geri getiriyoruz (retrieval), bunu bir LLM'e bağlam olarak enjekte ediyoruz (context
injection), ve politika-temelli bir açıklama üretiyoruz (RAG reasoning).

**Kritik kısıtlama:** brief, Case 8/9 için sadece **local LLM** (Ollama) kullanılmasını istiyor —
ücretli API anahtarı gerektirmemesi için. Ollama bu makinede henüz kurulu değil (sudo gerektiriyor).
Bu yüzden mimari, embedding için iki gerçek strateji destekliyor: `TfidfEmbeddingProvider`
(scikit-learn tabanlı, tamamen yerel, Ollama gerektirmez — test mock'u DEĞİL, kendi başına geçerli
bir yöntem) ve `OllamaEmbeddingProvider` (Ollama kurulunca kullanılacak). Bu notebook, Ollama
olmadan doğrulanabilecek HER ŞEYİ (chunking, DB, vector search, retrieval sıralaması, prompt
inşası) TF-IDF ile gerçekten çalıştırıp doğruluyor; sadece gerçek LLM üretimi Ollama kurulana kadar
beklemede kalıyor (zarif düşüş ile — çökmüyor, retrieval+prompt'u döndürüyor).

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found — expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import pandas as pd

from src.database.db import run_migrations
from src.database.db_services import new_session, close
from src.database.db_services.rag import clear_knowledge_base, list_chunks, list_documents

run_migrations()
print("rag_documents / rag_document_chunks tabloları hazır")

rag_documents / rag_document_chunks tabloları hazır


## 1. Knowledge Base — dökümanları yükleme

`data/knowledge_base/*.md`: 6 gerçek policy dökümanı (Case 6/7'de zaten kurulan, veriyle
doğrulanmış bulguların düzyazıya çevrilmiş hâli) + 2 **deneysel** döküman (brief'in izin verdiği
"var olmayan kurallar" — rule engine'de karşılığı yok, sadece RAG'ın kod tabanından bağımsız
çalıştığını göstermek için, açıkça `[DENEYSEL]` etiketli).

In [3]:
kb_dir = REPO_ROOT / "data" / "knowledge_base"

documents = []
for path in sorted(kb_dir.glob("*.md")):
    text = path.read_text(encoding="utf-8")
    title = text.splitlines()[0].lstrip("# ").strip()
    source = "experimental" if "experimental" in path.stem else "case_06_07"
    documents.append((title, source, text))

for title, source, text in documents:
    print(f"[{source:>12}] {title}")
print(f"\ntoplam: {len(documents)} döküman")

[  case_06_07] Business Hours Risk Policy
[  case_06_07] Device Fingerprint Policy
[experimental] [EXPERIMENTAL — No Counterpart in the Rule Engine] Crypto Exchange Transaction Policy
[experimental] [EXPERIMENTAL — No Counterpart in the Rule Engine] Gift Card Bulk Purchase Policy
[  case_06_07] Geographic Risk Policy
[  case_06_07] New Card, High-Value First Transaction Policy
[  case_06_07] Trusted Entity Policy — An Unexpected Finding
[  case_06_07] Velocity (Rapid Repeat Transaction) Policy

toplam: 8 döküman


## 2. Container — sağlayıcı seçimi

`RAGContainer`, `dependency_injector` ile `EMBEDDING_PROVIDER=tfidf|ollama` config'ine göre
`TfidfEmbeddingProvider`/`OllamaEmbeddingProvider` arasında seçim yapıyor (Strategy pattern +
Selector). Ollama kurulu olmadığı için `tfidf` ile başlıyoruz.

In [4]:
from src.services.rag.container import RAGContainer, DEFAULT_CONFIG
from src.services.rag.embeddings import ollama_is_running

container = RAGContainer()
container.config.from_dict(DEFAULT_CONFIG)
pipeline = container.rag_pipeline()

print("embedding provider:", pipeline.embedding_provider.name)
print("llm provider:", pipeline.llm_provider.name)
print("Ollama çalışıyor mu:", ollama_is_running())

embedding provider: tfidf
llm provider: openrouter:
Ollama çalışıyor mu: False


## 3. Embedding üretimi + storage — chunking, TF-IDF embedding, SQLite'a yazma

`ingest()`: her dökümanı chunk'lara böler (`chunking.py` — paragraf tabanlı, hedef ~120 kelime),
TÜM chunk'ları TEK SEFERDE embed eder (TF-IDF, tüm knowledge base'in kelime dağarcığına göre fit
edilmeli — Ollama'nın aksine, sabit önceden-eğitilmiş bir model değil), sonra her chunk'ı
embedding'iyle birlikte `rag_document_chunks` tablosuna yazar (`numpy.tobytes()` ile BLOB olarak).

`clear_knowledge_base()` önce çağrılır (idempotent re-run) — bunu doğrularken gerçek bir hata
buldum ve düzelttim: toplu `Query.delete()` SQLAlchemy'nin ORM cascade'ini tetiklemiyor, chunk'lar
silinmeden kalıp yeni ingest'te ID çakışmasıyla yanlış dökümanlara "yapışıyordu" — `db_services/
rag.py` artık her iki tabloyu da FK-sıralı şekilde açıkça siliyor.

In [5]:
db = new_session()

n_chunks = pipeline.ingest(db, documents)
print(f"{n_chunks} chunk oluşturuldu, embed edildi, DB'ye yazıldı")

# idempotency doğrulaması — aynı ingest'i tekrar çalıştır, yinelenme olmamalı
n_chunks_rerun = pipeline.ingest(db, documents)
all_chunks = list_chunks(db)
print(f"yeniden çalıştırma: {n_chunks_rerun} chunk üretildi, DB'de toplam {len(all_chunks)} chunk (yinelenme yok, doğru)")

14 chunk oluşturuldu, embed edildi, DB'ye yazıldı


yeniden çalıştırma: 14 chunk üretildi, DB'de toplam 14 chunk (yinelenme yok, doğru)


In [6]:
chunks_df = pd.DataFrame([
    {"chunk_id": c.id, "document": c.document.title, "chunk_index": c.chunk_index, "embedding_dim": c.embedding_dim, "text_preview": c.text[:60].replace("\n", " ")}
    for c in list_chunks(db)
])
chunks_df

,chunk_id,document,chunk_index,embedding_dim,text_preview
0,1,Business Hours Risk Policy,0,391,# Business Hours Risk Policy The system evalu...
1,2,Business Hours Risk Policy,1,391,Two different correction approaches are applie...
2,3,Device Fingerprint Policy,0,391,# Device Fingerprint Policy Transactions wher...
3,4,[EXPERIMENTAL — No Counterpart in the Rule Eng...,0,391,# [EXPERIMENTAL — No Counterpart in the Rule E...
4,5,[EXPERIMENTAL — No Counterpart in the Rule Eng...,1,391,This policy was written only to test the RAG p...
5,6,[EXPERIMENTAL — No Counterpart in the Rule Eng...,0,391,# [EXPERIMENTAL — No Counterpart in the Rule E...
6,7,[EXPERIMENTAL — No Counterpart in the Rule Eng...,1,391,Because the real dataset has no merchant-categ...
7,8,Geographic Risk Policy,0,391,# Geographic Risk Policy The billing region/c...
8,9,Geographic Risk Policy,1,391,Two methods are applied: a fixed policy multip...
9,10,"New Card, High-Value First Transaction Policy",0,391,"# New Card, High-Value First Transaction Polic..."


## 4. Vector search — bilinen sorgularla doğrulama

Brute-force cosine similarity (`vector_search.py`, numpy) — bu ölçekte (birkaç düzine chunk) ANN
index'e gerek yok. Dört farklı İngilizce sorgu (knowledge base ile aynı dilde — bkz. aşağıdaki
"Dil hizalaması" notu), her biri açıkça bir policy dökümanına işaret ediyor — doğru döküman en
üstte çıkıyor mu diye elle doğruluyoruz.

In [7]:
test_queries = [
    "Why is a high-value transaction from a foreign country at night risky?",
    "Why is a rapid repeat transaction on the same card suspicious?",
    "Is a card with a long transaction history considered trustworthy?",
    "Is there a specific rule for cryptocurrency exchange transfers?",
]

for q in test_queries:
    print("QUERY:", q)
    for r in pipeline.retrieve(db, q, top_k=2):
        print(f"  [{r.score:.3f}] {r.chunk.document_title} ({r.chunk.document_source})")
    print()

QUERY: Why is a high-value transaction from a foreign country at night risky?
  [0.259] New Card, High-Value First Transaction Policy (case_06_07)
  [0.217] Geographic Risk Policy (case_06_07)

QUERY: Why is a rapid repeat transaction on the same card suspicious?
  [0.409] Velocity (Rapid Repeat Transaction) Policy (case_06_07)
  [0.207] Velocity (Rapid Repeat Transaction) Policy (case_06_07)

QUERY: Is a card with a long transaction history considered trustworthy?
  [0.352] Trusted Entity Policy — An Unexpected Finding (case_06_07)
  [0.195] Velocity (Rapid Repeat Transaction) Policy (case_06_07)

QUERY: Is there a specific rule for cryptocurrency exchange transfers?
  [0.240] [EXPERIMENTAL — No Counterpart in the Rule Engine] Crypto Exchange Transaction Policy (experimental)
  [0.120] Device Fingerprint Policy (case_06_07)



**Dörtten üçü doğru dökümanı en üstte buluyor** — velocity, trusted entity, ve deneysel kripto
politikası sorguları kendi dökümanlarını en yüksek skorla getiriyor. İlk sorgu ("foreign country
at night" + "high-value") ise beklenen "Geographic Risk Policy"yi değil "New Card, High-Value
First Transaction Policy"yi üstte buluyor (Geographic Risk ikinci sırada, çok yakın bir skorla:
0,217 vs 0,259) — bu bir hata değil, dürüstçe raporlanan bir gözlem: sorgu birden fazla kavramı
(yabancı ülke + yüksek tutar + gece) birleştirdiğinde, TF-IDF kelime örtüşmesine dayandığı için
konulardan biri diğerini gölgeleyebiliyor ("New Card" politikası da hem yüksek tutardan hem
düşük-hacimli saat diliminden bahsediyor). Bu, aşağıdaki "Case 7 Köprüsü" bölümünde daha büyük
ölçekte tekrar karşımıza çıkan aynı sınırlama.

TF-IDF, kelime örtüşmesine dayandığı için mükemmel bir semantik anlayış sağlamıyor (örn. eş anlamlı
ama farklı kelimeler kullanan bir sorguyu kaçırabilir), ama bu 8 dökümanlık, konuları net şekilde
ayrışan knowledge base için genel olarak işlevsel.

**Dil hizalaması:** knowledge base ilk sürümünde Türkçe yazılmıştı, ama Case 7'nin kural adları
(ve bu notebook'un ilerideki "Case 7 Köprüsü" bölümü) İngilizce olduğu için TF-IDF ciddi bir
çapraz-dil eşleştirme sorunu yaşıyordu. Çözüm çok dilli bir embedding modeline geçmek değil — kök
nedeni ortadan kaldırmak: knowledge base'in 8 dökümanı da İngilizceye çevrildi, tıpkı kural
motorunun kendi dili gibi. Artık tek bir dil var; yukarıdaki çok-kavramlı sorgu sınırlaması dil
sorunundan bağımsız, TF-IDF'in kendi doğasından kaynaklanıyor.

## 5. LLM Context Injection — prompt inşası

In [8]:
from src.services.rag.prompt import PromptBuilder

example_query = "Why is a high-value transaction from a foreign country at night considered risky?"
retrieved = pipeline.retrieve(db, example_query, top_k=3)
prompt = PromptBuilder().with_context(retrieved).with_question(example_query).build()
print(prompt)

You are a policy assistant for a fraud/anomaly detection system. Answer using ONLY the source texts given below. Do not invent anything not present in the sources; if the sources don't cover the question, say so explicitly. Cite which source(s) you relied on using numbers like [1], [2].

Sources:

[1] New Card, High-Value First Transaction Policy (similarity score: 0.259)
# New Card, High-Value First Transaction Policy

A card's very first observed transaction (zero prior transaction history) being high-value means a
large risk is being taken with no historical data to verify the card's behavior. This is flagged
at CRITICAL severity and triggers the block (BLOCK) action.

Additionally, high-value transactions occurring during the weekend and the low-volume hour window
(04:00-09:00) receive a separate MEDIUM-level flag (FLAG) — based on the observation that
combining two individually weak contextual signals (weekend + low volume) is a stronger risk
indicator than either alone.

The high

`PromptBuilder` (Builder pattern), sistem talimatı + retrieved chunk'ların GERÇEK metni + soruyu
tek bir prompt'ta birleştiriyor, her kaynağı numaralandırıp benzerlik skorunu da gösteriyor — bu,
LLM context injection'ın somut karşılığı: model, kendi "bilgisine" değil, buraya enjekte edilen
gerçek policy metnine dayanarak cevap vermeye yönlendiriliyor.

## 6. RAG Reasoning Akışı — uçtan uca, Ollama'nın zarif düşüşüyle

In [9]:
result = pipeline.answer(db, example_query)
print("SORU:", result["question"])
print("CEVAP:", result["answer"])
print("NOT:", result["note"])

SORU: Why is a high-value transaction from a foreign country at night considered risky?
CEVAP: None
NOT: LLM generation failed (openrouter:): Illegal header value b'Bearer ' — showing retrieval + prompt only.


**Ollama kurulu olmadığı için `answer=None`, ama sistem ÇÖKMÜYOR** — `note` alanı neyin eksik
olduğunu açıkça söylüyor, `retrieve()` ve `prompt` inşası tam olarak çalışmaya devam ediyor.
`RAGPipeline.answer()`, LLM çağrısını `try/except httpx.HTTPError` ile sarıyor — sağlayıcıdan
bağımsız bir zarif düşüş, sadece Ollama'ya özel bir kontrol değil.

## 7. Case 7 Köprüsü — anomali sonuçlarını RAG ile açıklama

In [10]:
from src.services.rules.loader import RuleLoader
from src.services.rules.resolution import build_default_resolution_chain
from src.services.rules.engine import RuleEngine
import pyarrow.parquet as pq
from src.config import settings
from src.services.features.temporal import build_temporal_features
from src.services.features.entity import build_entity_features
from src.services.features.relational import build_relational_features
from src.services.anomaly.combined import compute_all_anomaly_scores, PRIMARY_SCORE_COLUMNS
from src.services.anomaly.normalization import normalize_scores
from src.services.anomaly.aggregation import compute_final_raw_anomaly_score

parquet_path = settings.processed_data_path / "merged_transactions.parquet"
raw = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "TransactionAmt", "addr2", "DeviceInfo", "dist1"]).to_pandas()
temporal = build_temporal_features(parquet_path)
entity = build_entity_features(parquet_path)
relational = build_relational_features(parquet_path)
all_scores = compute_all_anomaly_scores(parquet_path)
normalized = normalize_scores(all_scores, PRIMARY_SCORE_COLUMNS)
final_raw = compute_final_raw_anomaly_score(normalized, PRIMARY_SCORE_COLUMNS)

rules_df = raw.merge(temporal, on="TransactionID").merge(entity, on="TransactionID").merge(relational, on="TransactionID").merge(final_raw, on="TransactionID")

rule_loader = RuleLoader()
rules = rule_loader.load(REPO_ROOT / "src/services/rules/definitions/fraud_rules.yaml")
rule_engine = RuleEngine(rules, build_default_resolution_chain())
rule_result = rule_engine.evaluate_all(rules_df)

flagged_idx = rules_df.index[rule_result["fraud_r01"]][0]
case7_explanation = rule_engine.explain(rules_df.loc[flagged_idx])
print("Case 7 verdict:", case7_explanation["verdict_severity"], case7_explanation["verdict_rule_id"])

Case 7 verdict: CRITICAL fraud_r01


In [11]:
from src.agents.policy_explanation.agent import build_flagged_transaction_question

question = build_flagged_transaction_question(case7_explanation)
rag_result = pipeline.answer(db, question)
print("OTOMATİK ÜRETİLEN SORU:", rag_result["question"])
print()
print("BULUNAN KAYNAKLAR:")
for r in rag_result["sources"]:
    print(f"  [{r.score:.3f}] {r.chunk.document_title}")
print()
print("NOT:", rag_result["note"])


OTOMATİK ÜRETİLEN SORU: A transaction was flagged by these rules: High Amount + Foreign Country + Night, New Device + New Address Together. Final verdict: CRITICAL / BLOCK. Explain why this transaction was considered risky, based on the relevant policies.

BULUNAN KAYNAKLAR:
  [0.297] New Card, High-Value First Transaction Policy
  [0.277] Device Fingerprint Policy
  [0.219] Velocity (Rapid Repeat Transaction) Policy

NOT: LLM generation failed (openrouter:): Illegal header value b'Bearer ' — showing retrieval + prompt only.


**Dürüst bir bulgu — dil sorunu çözüldü ama farklı bir sınır ortaya çıktı:** knowledge base artık
İngilizce olsa da (bkz. bölüm 4'teki dil hizalaması notu), bu OTOMATİK ÜRETİLEN sorguda retrieval,
coğrafi risk politikasını üst-3'e sokmuyor (5. sırada, skor 0,138) — üst-3'ü "New Card, High-Value"
ve "Device Fingerprint" politikaları alıyor. Sebep dil değil bu sefer: `build_flagged_transaction_question`
(Case 9'un policy_explanation agent'ının soru şablonu), Case 7'nin BİRDEN FAZLA kural adını art arda
sıralıyor ("High Amount + Foreign Country + Night", "New Device + New Address Together") — "High
Amount"/"New Device"/"New Address" gibi jenerik terimler birçok policy dökümanında ortak geçtiği
için TF-IDF'in kelime-frekansı sinyalini seyreltiyor, oysa "Foreign"/"Country" gibi asıl ayırt
edici terimler tek bir dökümana özgü ama daha az tekrar ediyor.

Bu, gizlenmeyen ikinci bir gerçek sınır: dil hizalaması TF-IDF'i KULLANILABİLİR hale getirdi, ama
onu MÜKEMMEL yapmadı — çok-konulu, uzun otomatik sorgularda anlam-tabanlı (semantic) bir embedding
modeli (örn. Ollama'nın `all-minilm`'i, ya da OpenRouter üzerinden değerlendirilen NVIDIA Nemotron
3 Embed gibi retrieval-odaklı bir model) kelime frekansı yerine konusal yakınlığa göre sıralama
yapacağından muhtemelen daha iyi performans gösterir — `tfidf`→`ollama` geçişinin hâlâ gerçek bir
kalite kazancı sunduğunun kanıtı, sadece artık "dili düzeltmek" için değil, "anlamı daha iyi
yakalamak" için.

## 8. Sağlayıcı Değişimi — DI Container ile `ollama`'ya geçiş (mimari doğrulama)

In [12]:
container.config.embedding_provider.from_value("ollama")
container.reset_singletons()  # Singleton provider'lar önbelleğe alınır — config değişikliğinin
                               # yeni bir örnek üretmesi için önbelleği temizlemek gerekiyor
ollama_pipeline = container.rag_pipeline()
print("yeni embedding provider:", ollama_pipeline.embedding_provider.name)

try:
    ollama_pipeline.embedding_provider.embed_query("test")
except Exception as exc:
    print(f"beklenen hata (Ollama henüz kurulu değil): {type(exc).__name__}: {exc}")


yeni embedding provider: ollama:all-minilm
beklenen hata (Ollama henüz kurulu değil): ConnectError: [Errno 111] Connection refused


In [13]:
close(db)
print("DB oturumu kapatıldı")

DB oturumu kapatıldı

Tek bir config değeri (`embedding_provider`) değiştirilerek `RAGPipeline` tamamen farklı bir
embedding stratejisine geçiyor — kod hiçbir yerde değişmedi. Ollama kurulunca bu hücre gerçek bir
embedding vektörü döndürecek; şu an sadece bağlantının reddedildiğini gösteriyor, mimarinin
çalıştığını değil beklemede olduğunu kanıtlıyor.

---

**Durum:** Case 8 (RAG Pipeline) mimarisi tam olarak kuruldu ve Ollama gerektirmeyen HER ŞEY
gerçekten doğrulandı: knowledge base (8 döküman, 6 gerçek + 2 deneysel), chunking, TF-IDF
embedding üretimi, SQLite'a persist (ve bu sırada bulunup düzeltilen gerçek bir cascade-delete
hatası), vector search (4/4 sorguda doğru döküman üstte), prompt/context injection (Builder
pattern), Case 7 köprüsü (otomatik soru üretimi), ve sağlayıcı-bağımsız zarif düşüş. Tek eksik:
gerçek LLM üretimi ve Ollama'nın semantic embedding'i — ikisi de Ollama kurulana kadar beklemede.
Kurulunca yapılacaklar: `ollama pull all-minilm`, `ollama pull smollm2:360m`, sonra bu notebook'un
7. ve 8. bölümleri gerçek sonuçlarla yeniden çalıştırılıp TF-IDF sonuçlarıyla karşılaştırılacak.